# Conditional Anomaly Detection

Our previous feature engineering experiments did not improve CatBoost much. Here, we test whether **unusual relationships between features** carry additional information about fraud.

We will predict `velocity_score` from the other features and use the prediction error as a new feature. We will compare the result with our original CatBoost baseline using the **same CV folds**.

## 1. Load Data

Use the original training data and saved baseline OOF predictions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

root = Path("../playground-series-s6e11")
train_df = pd.read_csv(root / "train.csv")
baseline_df = pd.read_csv("baseline_artifacts/baseline_oof.csv")

SEED = 42
N_SPLITS = 5
INNER_SPLITS = 3
TARGET = "is_fraud"
PREDICTED_FEATURE = "velocity_score"

print("Train:", train_df.shape)
print("Baseline OOF:", baseline_df.shape)

Train: (699960, 26)
Baseline OOF: (699960, 4)


## 2. Prepare Features

Match the preprocessing used in the CatBoost baseline. `velocity_score` will be excluded only when training the regression model.

In [2]:
feature_cols = [c for c in train_df.columns if c not in ["id", TARGET]]
X = train_df[feature_cols].copy()
y = train_df[TARGET].astype(int)

cat_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
bool_features = X.select_dtypes(include=["bool", "boolean"]).columns.tolist()

for col in cat_features:
    X[col] = X[col].astype("string").fillna("__MISSING__").astype(str)
for col in bool_features:
    X[col] = X[col].astype("Int64").astype(float)

reg_features = [c for c in feature_cols if c != PREDICTED_FEATURE]
X_reg = X[reg_features]
velocity = X[PREDICTED_FEATURE]

baseline = train_df[["id", TARGET]].merge(
    baseline_df[["id", "fold", "oof_pred"]], on="id", how="left", validate="one_to_one"
)
assert baseline["oof_pred"].notna().all()
print("Baseline OOF AUC:", round(roc_auc_score(y, baseline["oof_pred"]), 6))

Baseline OOF AUC: 0.946094


## 3. Experiment: Velocity Residual

For each outer fold, we predict `velocity_score` on the training rows using inner OOF regression, and on validation rows using a regression model trained only on the outer training fold.

We then add `velocity_residual = abs(actual - predicted)` to the original features and train CatBoost. This avoids using a row to train the regression model that generates its own residual.

In [3]:
reg_params = {
    "iterations": 500,
    "depth": 5,
    "learning_rate": 0.05,
    "loss_function": "RMSE",
    "random_seed": SEED,
    "verbose": False,
    "thread_count": -1,
}

clf_params = {
    "iterations": 3000,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": SEED,
    "verbose": False,
    "thread_count": -1,
}

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
residual_oof = np.full(len(X), np.nan)
new_oof = np.full(len(X), np.nan)
fold_results = []

In [4]:
for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    # Inner OOF residuals for the outer training rows
    inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    inner_pred = np.full(len(train_idx), np.nan)

    for inner_train, inner_valid in inner_cv.split(train_idx):
        reg = CatBoostRegressor(**reg_params)
        reg.fit(
            X_reg.iloc[train_idx[inner_train]],
            velocity.iloc[train_idx[inner_train]],
            cat_features=cat_features,
        )
        inner_pred[inner_valid] = reg.predict(X_reg.iloc[train_idx[inner_valid]])

    # Residuals for outer validation rows: never train on these rows
    reg = CatBoostRegressor(**reg_params)
    reg.fit(X_reg.iloc[train_idx], velocity.iloc[train_idx], cat_features=cat_features)
    valid_pred = reg.predict(X_reg.iloc[valid_idx])

    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    X_train["velocity_residual"] = np.abs(velocity.iloc[train_idx].to_numpy() - inner_pred)
    X_valid["velocity_residual"] = np.abs(velocity.iloc[valid_idx].to_numpy() - valid_pred)

    clf = CatBoostClassifier(**clf_params)
    clf.fit(
        X_train, y.iloc[train_idx],
        cat_features=cat_features,
        eval_set=(X_valid, y.iloc[valid_idx]),
        early_stopping_rounds=100,
        use_best_model=True,
    )

    pred = clf.predict_proba(X_valid)[:, 1]
    new_oof[valid_idx] = pred
    residual_oof[valid_idx] = X_valid["velocity_residual"].to_numpy()

    baseline_fold_auc = roc_auc_score(y.iloc[valid_idx], baseline["oof_pred"].iloc[valid_idx])
    new_fold_auc = roc_auc_score(y.iloc[valid_idx], pred)
    fold_results.append({
        "fold": fold,
        "baseline_auc": baseline_fold_auc,
        "residual_auc": new_fold_auc,
        "delta": new_fold_auc - baseline_fold_auc,
    })
    print(f"Fold {fold}: AUC={new_fold_auc:.6f}, delta={new_fold_auc - baseline_fold_auc:+.6f}")

Fold 1: AUC=0.944323, delta=-0.000239
Fold 2: AUC=0.947329, delta=-0.000133
Fold 3: AUC=0.950822, delta=+0.000270
Fold 4: AUC=0.945898, delta=-0.000454
Fold 5: AUC=0.941318, delta=-0.000573


## 4. Results

First check whether the residual has individual ranking power, then whether it improves CatBoost beyond the original features.

In [5]:
results_df = pd.DataFrame(fold_results)
display(results_df)

baseline_auc = roc_auc_score(y, baseline["oof_pred"])
residual_auc = roc_auc_score(y, residual_oof)
new_auc = roc_auc_score(y, new_oof)

print(f"Residual univariate AUC: {residual_auc:.6f}")
print(f"Baseline OOF AUC:       {baseline_auc:.6f}")
print(f"With residual OOF AUC:  {new_auc:.6f}")
print(f"Delta:                  {new_auc - baseline_auc:+.6f}")

,fold,baseline_auc,residual_auc,delta
0,1,0.944562,0.944323,-0.000239
1,2,0.947463,0.947329,-0.000133
2,3,0.950552,0.950822,0.000270
3,4,0.946352,0.945898,-0.000454
4,5,0.941891,0.941318,-0.000573


Residual univariate AUC: 0.531739
Baseline OOF AUC:       0.946094
With residual OOF AUC:  0.945899
Delta:                  -0.000195


## 5. Save Results

Save OOF predictions so we can revisit the experiment without retraining.

In [6]:
output_dir = Path("anomaly_artifacts")
output_dir.mkdir(exist_ok=True)

pd.DataFrame({
    "id": train_df["id"],
    "is_fraud": y,
    "velocity_residual": residual_oof,
    "oof_pred": new_oof,
}).to_csv(output_dir / "velocity_residual_oof.csv", index=False)

results_df.to_csv(output_dir / "velocity_residual_folds.csv", index=False)
print("Saved to:", output_dir)

Saved to: anomaly_artifacts


## 6. Absolute Residual Findings

The absolute velocity residual achieved a univariate ROC-AUC of ***0.531739***, suggesting a weak fraud signal.

However, adding it to CatBoost reduced OOF ROC-AUC from ***0.946094*** to ***0.945899***. Performance decreased on four out of five folds.

The residual may contain information already captured by the original features. We will not keep it in the model.

One possible limitation is that taking the absolute value removes the direction of the prediction error. We will test this next.

## 7. Signed Velocity Residual

Our previous experiment used the absolute difference between actual and predicted velocity.

This time, we will preserve the direction of the difference to check whether higher-than-expected velocity is associated with fraud.

In [7]:
signed_oof = np.full(len(X), np.nan)
signed_residual_oof = np.full(len(X), np.nan)
signed_fold_results = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    inner_cv = KFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED
    )

    inner_pred = np.full(len(train_idx), np.nan)

    for inner_train, inner_valid in inner_cv.split(train_idx):
        reg = CatBoostRegressor(**reg_params)

        reg.fit(
            X_reg.iloc[train_idx[inner_train]],
            velocity.iloc[train_idx[inner_train]],
            cat_features=cat_features
        )

        inner_pred[inner_valid] = reg.predict(
            X_reg.iloc[train_idx[inner_valid]]
        )

    reg = CatBoostRegressor(**reg_params)
    reg.fit(
        X_reg.iloc[train_idx],
        velocity.iloc[train_idx],
        cat_features=cat_features
    )

    valid_pred = reg.predict(X_reg.iloc[valid_idx])

    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()

    X_train["velocity_signed_residual"] = (
        velocity.iloc[train_idx].to_numpy() - inner_pred
    )

    X_valid["velocity_signed_residual"] = (
        velocity.iloc[valid_idx].to_numpy() - valid_pred
    )

    clf = CatBoostClassifier(**clf_params)

    clf.fit(
        X_train,
        y.iloc[train_idx],
        cat_features=cat_features,
        eval_set=(X_valid, y.iloc[valid_idx]),
        early_stopping_rounds=100,
        use_best_model=True
    )

    pred = clf.predict_proba(X_valid)[:, 1]

    signed_oof[valid_idx] = pred
    signed_residual_oof[valid_idx] = (
        X_valid["velocity_signed_residual"].to_numpy()
    )

    auc = roc_auc_score(y.iloc[valid_idx], pred)
    baseline_fold_auc = roc_auc_score(
        y.iloc[valid_idx],
        baseline["oof_pred"].iloc[valid_idx]
    )

    signed_fold_results.append({
        "fold": fold,
        "auc": auc,
        "delta": auc - baseline_fold_auc
    })

    print(f"Fold {fold}: AUC={auc:.6f}")

Fold 1: AUC=0.944155
Fold 2: AUC=0.947418
Fold 3: AUC=0.950760
Fold 4: AUC=0.946422
Fold 5: AUC=0.941566


## 8. Signed Residual Results

We will compare the signed residual with the absolute residual and the original `CatBoost` baseline.

We will also check whether preserving the direction of the error makes the residual more informative on its own.

In [8]:
signed_auc = roc_auc_score(y, signed_residual_oof)
signed_model_auc = roc_auc_score(y, signed_oof)

comparison = pd.DataFrame({
    "experiment": [
        "Baseline",
        "Absolute residual",
        "Signed residual"
    ],
    "oof_auc": [
        baseline_auc,
        new_auc,
        signed_model_auc
    ]
})

comparison["delta"] = comparison["oof_auc"] - baseline_auc

print(f"Signed residual univariate AUC: {signed_auc:.6f}")

display(comparison)
display(pd.DataFrame(signed_fold_results))

Signed residual univariate AUC: 0.635527


,experiment,oof_auc,delta
0,Baseline,0.946094,0.000000
1,Absolute residual,0.945899,-0.000195
2,Signed residual,0.946008,-0.000086


,fold,auc,delta
0,1,0.944155,-0.000407
1,2,0.947418,-0.000044
2,3,0.950760,0.000208
3,4,0.946422,0.000070
4,5,0.941566,-0.000324


## 9. Final Findings

The signed residual was more informative than the absolute residual:

- Absolute residual AUC: ***0.531739***
- Signed residual AUC: ***0.635527***

This suggests that the direction of the velocity prediction error matters. Higher-than-expected velocity tends to be more associated with fraud.

However, neither feature improved `CatBoost`

The signed residual performed better than the absolute version, but still fell slightly below the baseline.

This suggests that `CatBoost` may already capture much of this information from the original features.